In [1]:
# YOLO - A1. Ultralytics (YOLOv8) 설치
!pip -q install ultralytics

In [26]:
# YOLO - A2. (Validation 전용) 데이터 변환 재실행

import os
import json
from tqdm.notebook import tqdm
from PIL import Image
import shutil

# --- 경로 설정 ---
FINAL_DATA_DIR = os.path.join(os.path.expanduser("~"), "projects/fashion_ai/data/deepfashion2_final")

# --- 변환 함수 (이전과 동일) ---
def convert_to_yolo(image_width, image_height, box):
    x_min, y_min, w, h = box
    x_center = (x_min + w / 2) / image_width
    y_center = (y_min + h / 2) / image_height
    width_norm = w / image_width
    height_norm = h / image_height
    return x_center, y_center, width_norm, height_norm

# --- 'validation' 데이터셋에 대해서만 변환 작업 수행 ---
split = 'val' # YOLO 표준 폴더 이름 사용
print(f"Re-processing '{split}' set...")

image_dir = os.path.join(FINAL_DATA_DIR, 'images', split)
ann_dir = os.path.join(FINAL_DATA_DIR, 'annos', split) # 임시 annos 폴더 경로
label_dir = os.path.join(FINAL_DATA_DIR, 'labels', split)
os.makedirs(label_dir, exist_ok=True)

# 만약을 위해 annos 폴더가 없으면 다시 추출
if not os.path.isdir(ann_dir):
    print(f"Extracting 'annos' for {split} set...")
    os.system(f"unzip /mnt/a/Data/DeepFashion2/archive.zip 'DeepFashion2/deepfashion2_original_images/validation/annos/*' -d {FINAL_DATA_DIR}/")
    # 추출된 폴더를 올바른 위치로 이동
    shutil.move(os.path.join(FINAL_DATA_DIR, 'DeepFashion2/deepfashion2_original_images/validation/annos'), ann_dir)
    shutil.rmtree(os.path.join(FINAL_DATA_DIR, 'DeepFashion2')) # 불필요한 상위 폴더 삭제

for json_file in tqdm(os.listdir(ann_dir)):
    if not json_file.endswith('.json'): continue
    json_path = os.path.join(ann_dir, json_file)
    image_path = os.path.join(image_dir, os.path.splitext(json_file)[0] + '.jpg')
    
    try:
        with Image.open(image_path) as img: img_width, img_height = img.size
        with open(json_path, 'r') as f: ann = json.load(f)
        label_file_path = os.path.join(label_dir, os.path.splitext(json_file)[0] + '.txt')
        with open(label_file_path, 'w') as f_label:
            for key, item_data in ann.items():
                if key.startswith('item'):
                    yolo_class_id = item_data['category_id'] - 1
                    yolo_box = convert_to_yolo(img_width, img_height, item_data['bounding_box'])
                    # 좌표 보정
                    yolo_box = [max(0.0, min(1.0, v)) for v in yolo_box]
                    f_label.write(f"{yolo_class_id} {' '.join(map(str, yolo_box))}\n")
    except:
        continue

# 정리 작업
shutil.rmtree(os.path.join(FINAL_DATA_DIR, 'annos')) # 임시 annos 폴더 삭제

print(f"\nValidation labels have been successfully created in: {label_dir}")

Re-processing 'val' set...
Extracting 'annos' for val set...
Archive:  /mnt/a/Data/DeepFashion2/archive.zip
  inflating: /home/epistao/projects/fashion_ai/data/deepfashion2_final/DeepFashion2/deepfashion2_original_images/validation/annos/000001.json  
  inflating: /home/epistao/projects/fashion_ai/data/deepfashion2_final/DeepFashion2/deepfashion2_original_images/validation/annos/000002.json  
  inflating: /home/epistao/projects/fashion_ai/data/deepfashion2_final/DeepFashion2/deepfashion2_original_images/validation/annos/000003.json  
  inflating: /home/epistao/projects/fashion_ai/data/deepfashion2_final/DeepFashion2/deepfashion2_original_images/validation/annos/000004.json  
  inflating: /home/epistao/projects/fashion_ai/data/deepfashion2_final/DeepFashion2/deepfashion2_original_images/validation/annos/000005.json  
  inflating: /home/epistao/projects/fashion_ai/data/deepfashion2_final/DeepFashion2/deepfashion2_original_images/validation/annos/000006.json  
  inflating: /home/epistao/p

  0%|          | 0/32153 [00:00<?, ?it/s]


Validation labels have been successfully created in: /home/epistao/projects/fashion_ai/data/deepfashion2_final/labels/val


In [27]:
# YOLO - A3. (최종 수정) 데이터셋 설정 파일(YAML) 생성

import os

# --- 기본 경로 설정 ---
HOME_DIR = os.path.expanduser("~")
PROJECT_DIR = os.path.join(HOME_DIR, "projects/fashion_ai")
FINAL_DATA_DIR = os.path.join(PROJECT_DIR, 'data', 'deepfashion2_final')
YAML_PATH = os.path.join(PROJECT_DIR, 'deepfashion2.yaml')

# --- 새로운 폴더 구조에 맞는 YAML 내용 ---
yaml_content = f"""
path: {FINAL_DATA_DIR}
train: images/train
val: images/val

names:
  0: 'short sleeve top'
  1: 'long sleeve top'
  2: 'short sleeve outwear'
  3: 'long sleeve outwear'
  4: 'vest'
  5: 'sling'
  6: 'shorts'
  7: 'trousers'
  8: 'skirt'
  9: 'short sleeve dress'
  10: 'long sleeve dress'
  11: 'vest dress'
  12: 'sling dress'
"""

# --- YAML 파일 덮어쓰기 ---
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print(f"YOLO 데이터셋 설정 파일이 (최종 폴더 구조 버전으로) 덮어쓰기 되었습니다.")
print(f"파일 위치: {YAML_PATH}")

YOLO 데이터셋 설정 파일이 (최종 폴더 구조 버전으로) 덮어쓰기 되었습니다.
파일 위치: /home/epistao/projects/fashion_ai/deepfashion2.yaml


In [28]:
# YOLO - B1. (수정됨) YOLOv8 모델 훈련 실행

import os

# --- 기본 경로 설정 ---
HOME_DIR = os.path.expanduser("~")
PROJECT_DIR = os.path.join(HOME_DIR, "projects/fashion_ai")
YAML_PATH = os.path.join(PROJECT_DIR, 'deepfashion2.yaml')

# --- YOLOv8 훈련 명령어 수정 ---
# 맨 끝에 workers=0 옵션을 추가하여 데이터 로딩 안정성을 확보합니다.
!yolo train data={YAML_PATH} model=yolov8n.pt epochs=3 imgsz=640 batch=8

New https://pypi.org/project/ultralytics/8.3.214 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.213 🚀 Python-3.11.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/epistao/projects/fashion_ai/deepfashion2.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, na